## Chapter 6: Tools for Model Training and Experimenting/Project:
* Favorita Store Sales - Time Series Forecasting/CH6-01-Building Favorita Baseline Model.py

In [0]:
%pip install -q databricks-feature-engineering
dbutils.library.restartPython()

In [0]:
from databricks.feature_engineering import FeatureEngineeringClient, FeatureLookup
from sklearn.model_selection import TimeSeriesSplit
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_squared_error
from sklearn.ensemble import RandomForestRegressor
import pyspark.pandas as ps
import numpy as np
import mlflow

spark.sql("USE CATALOG porya_catalog")
spark.sql("USE SCHEMA default")

fe = FeatureEngineeringClient()
training_features = "training_dataset"
raw_data_table = "train_set"
label_name = "sales"
time_column = "date"


raw_data = sql(f"SELECT * FROM {raw_data_table}")

display(raw_data.take(10))

# COMMAND ----------

store_data = spark.table("favorita_stores")

display(store_data.take(10))

# COMMAND ----------

store_data = spark.table("oil_10d_lag_ft")

display(store_data.take(10))

In [0]:
model_feature_lookups = [
    FeatureLookup(
      table_name="oil_10d_lag_ft",
      lookup_key="date",
      feature_names="lag10_oil_price"
    ),
    FeatureLookup(
      table_name="store_holidays_ft",
      lookup_key=["date","store_nbr"]
    ),
    FeatureLookup(
      table_name="stores_ft",
      lookup_key="store_nbr",
      feature_names=["store_cluster","store_type"]
    ),
]

In [0]:
# DBTITLE 1,Create the training set
training_set = fe.create_training_set(
    df=raw_data,
    feature_lookups=model_feature_lookups,
    label=label_name,
)
training_df = training_set.load_df()
# DBTITLE 1,Save as table
training_df.write.mode("overwrite").saveAsTable("training_data")

# COMMAND ----------
display(training_df)

In [0]:
import databricks.automl as automl

automl_data = training_df.filter("date > '2016-12-31'")

summary = automl.regress(automl_data, 
                          target_col=label_name,
                          time_col="date",
                          timeout_minutes=10,
                          exclude_cols=['id']
                          )

In [0]:
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
cluster = w.clusters.get("0317-131601-yqfdnb1u")

print(f"Name: {cluster.cluster_name}")
print(f"State: {cluster.state}")
print(f"Runtime: {cluster.spark_version}")
print(f"Access Mode: {cluster.data_security_mode}")
print(f"Single User: {cluster.single_user_name}")
print(f"Node Type: {cluster.node_type_id}")
print(f"Autoscale: {cluster.autoscale}")
print(f"Num Workers: {cluster.num_workers}")